# 02 — Construct Grouped Mini-Batches (FAISS-based)

**Goal:** Build a deterministic, research-grade grouped mini-batch sampler for instruction fine-tuning.

This notebook:
1. Loads `bootstrap_manifest.json`
2. Loads dataset from Drive
3. Loads embeddings + FAISS index bundle
4. Precomputes neighbor lists (top-k) per example (cached)
5. Builds a **GroupedBatchSampler** that:
   - anchors each batch on a seed example
   - fills remaining slots with nearest neighbors
   - avoids duplicates
   - enforces deterministic behavior via a seed
6. Provides a debug DataLoader to inspect batches

Output artifacts written to Drive:
- `neighbors_topk.npy` (indices)
- `neighbors_scores.npy` (similarity scores)
- `grouped_batches.jsonl` (optional debug export)

In [ ]:
import os
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print("Google Drive already mounted.")

Mounted at /content/drive


In [ ]:
!pip install -q faiss-cpu sentence-transformers datasets tqdm  # faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 51.5 MB/s eta 0:00:00


In [ ]:
import os
import json
import time
import math
import random
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional

import numpy as np
from tqdm.auto import tqdm

import faiss
from datasets import load_from_disk

## Load Manifest + Resolve Paths

In [ ]:
ROOT = "/content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research"
MANIFEST_PATH = f"{ROOT}/manifests/bootstrap_manifest.json"

if not os.path.exists(MANIFEST_PATH):
    raise FileNotFoundError(
        f"bootstrap_manifest.json not found at: {MANIFEST_PATH}\n"
        "Run 00_bootstrap_downloads.ipynb first."
    )

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

DIRS = manifest["dirs"]

def _safe_makedirs(p: str):
    os.makedirs(p, exist_ok=True)

def _exists_nonempty(path: str) -> bool:
    return os.path.exists(path) and (
        (os.path.isdir(path) and len(os.listdir(path)) > 0)
        or (os.path.isfile(path) and os.path.getsize(path) > 0)
    )

print("Loaded manifest created_at_utc:", manifest.get("created_at_utc"))
print("ROOT:", manifest.get("root", ROOT))

Loaded manifest created_at_utc: 2026-03-05T17:21:43Z
ROOT: /content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research


## Config

- `DATASET_NAME`: should match what you used for embeddings+FAISS
- `BUNDLE_NAME`: `"{DATASET_NAME}__{EMBED_MODEL_DIRNAME}"`
- `TOP_K`: neighbors precomputed per example
- `BATCH_SIZE`: size of grouped mini-batch
- `SEED`: controls determinism

In [ ]:
FORCE_REBUILD = False

DATASET_NAME = "dolly_small_1k"   # or "dolly_15k"
SPLIT_NAME = "train"
TEXT_FIELD = "instruction"

EMBED_MODEL_DIRNAME = "all-MiniLM-L6-v2"
BUNDLE_NAME = f"{DATASET_NAME}__{EMBED_MODEL_DIRNAME}"

TOP_K = 32          # number of neighbors stored per example
BATCH_SIZE = 8      # your training batch size later
SEED = 42           # determinism
EXPORT_DEBUG_BATCHES = True  # writes a jsonl with a few example batches

assert TOP_K >= BATCH_SIZE, "TOP_K should be >= BATCH_SIZE to fill a batch."

random.seed(SEED)
np.random.seed(SEED)

## 4) Load Dataset + FAISS Bundle

In [ ]:
dataset_path = f"{DIRS['shared_datasets_raw']}/{DATASET_NAME}"
bundle_dir = f"{DIRS['shared_indexes_faiss']}/{BUNDLE_NAME}"

emb_path = f"{bundle_dir}/embeddings.npy"
faiss_index_path = f"{bundle_dir}/faiss.index"
id_map_path = f"{bundle_dir}/id_map.json"
meta_path = f"{bundle_dir}/meta.json"

for p in [dataset_path, bundle_dir, emb_path, faiss_index_path, id_map_path, meta_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing required path: {p}")

ds = load_from_disk(dataset_path)
split = ds[SPLIT_NAME]

X = np.load(emb_path).astype("float32")
index = faiss.read_index(faiss_index_path)

with open(id_map_path, "r", encoding="utf-8") as f:
    id_map = json.load(f)["id_map"]

with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

print("Dataset rows:", len(split))
print("Embeddings shape:", X.shape)
print("FAISS ntotal:", index.ntotal)
print("Meta:", meta)

Dataset rows: 1000
Embeddings shape: (1000, 384)
FAISS ntotal: 1000
Meta: {'created_at_utc': '2026-03-05T17:39:46Z', 'dataset_name': 'dolly_small_1k', 'split': 'train', 'text_field': 'instruction', 'embed_model': 'all-MiniLM-L6-v2', 'n_rows': 1000, 'dim': 384, 'normalized': True, 'dtype': 'float32'}


## 5) Precompute top-k Neighbors (Top-K) (cached)

We compute for each row `i`:
- `neighbors_idx[i]`: array of top-k neighbor indices (including itself at rank 0)
- `neighbors_score[i]`: similarity scores

Saved to Drive so training notebooks can reuse it without recomputing.

In [ ]:
neighbors_idx_path = f"{bundle_dir}/neighbors_topk_idx.npy"
neighbors_score_path = f"{bundle_dir}/neighbors_topk_scores.npy"

need_neighbors = FORCE_REBUILD or (not _exists_nonempty(neighbors_idx_path)) or (not _exists_nonempty(neighbors_score_path))

if need_neighbors:
    n = X.shape[0]
    neighbors_idx = np.empty((n, TOP_K), dtype=np.int32)
    neighbors_score = np.empty((n, TOP_K), dtype=np.float32)

    # Query in batches for speed
    qbs = 512
    for i in tqdm(range(0, n, qbs), desc="FAISS top-k"):
        q = X[i:i+qbs]
        scores, idxs = index.search(q, TOP_K)
        neighbors_idx[i:i+qbs] = idxs.astype(np.int32)
        neighbors_score[i:i+qbs] = scores.astype(np.float32)

    np.save(neighbors_idx_path, neighbors_idx)
    np.save(neighbors_score_path, neighbors_score)

    print("Saved:", neighbors_idx_path)
    print("Saved:", neighbors_score_path)
else:
    neighbors_idx = np.load(neighbors_idx_path)
    neighbors_score = np.load(neighbors_score_path)
    print("Loaded cached neighbors:", neighbors_idx.shape)

FAISS top-k:   0%|          | 0/2 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/indexes/faiss/dolly_small_1k__all-MiniLM-L6-v2/neighbors_topk_idx.npy
Saved: /content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/indexes/faiss/dolly_small_1k__all-MiniLM-L6-v2/neighbors_topk_scores.npy


## 6) GroupedBatchSampler (Deterministic) (core logic)

Batch construction strategy:
- iterate over a shuffled list of anchor indices (seed-controlled)
- for each anchor `a`:
  - start batch = [a]
  - iterate neighbors of `a` in similarity order
  - add neighbor if not used in this batch and not globally exhausted (optional)
  - stop when batch_size reached
- remaining unassigned anchors still appear later, so coverage stays good

This gives you semantic coherence without requiring a global O(n²) ordering.

In [ ]:
@dataclass
class GroupedBatchSamplerConfig:
    batch_size: int
    seed: int
    drop_last: bool = True
    avoid_duplicate_within_batch: bool = True

class GroupedBatchSampler:
    """
    Deterministic grouped batching using precomputed neighbor lists.

    Notes:
    - This sampler returns lists of indices (batches).
    - It does NOT modify dataset order globally; it constructs batches on demand.
    - Determinism comes from the seed + stable neighbor arrays.
    """
    def __init__(self, neighbors_idx: np.ndarray, cfg: GroupedBatchSamplerConfig):
        self.neighbors_idx = neighbors_idx
        self.cfg = cfg
        self.n = neighbors_idx.shape[0]

    def build_batches(self) -> List[List[int]]:
        rng = random.Random(self.cfg.seed)
        anchors = list(range(self.n))
        rng.shuffle(anchors)

        batches: List[List[int]] = []
        used_global = set()  # optional: prevents repeats across batches (strong constraint)

        for a in anchors:
            if a in used_global:
                continue

            batch = [a]
            used_in_batch = {a}

            # Iterate over neighbors in rank order
            for nb in self.neighbors_idx[a]:
                if len(batch) >= self.cfg.batch_size:
                    break

                if self.cfg.avoid_duplicate_within_batch and nb in used_in_batch:
                    continue

                # Avoid repeats across batches for cleaner coverage
                if nb in used_global:
                    continue

                batch.append(int(nb))
                used_in_batch.add(int(nb))

            if len(batch) == self.cfg.batch_size:
                batches.append(batch)
                for x in batch:
                    used_global.add(x)
            else:
                # If we can't fill the batch, either drop it or keep it (config)
                if not self.cfg.drop_last and len(batch) > 0:
                    batches.append(batch)
                    for x in batch:
                        used_global.add(x)

        return batches

cfg = GroupedBatchSamplerConfig(batch_size=BATCH_SIZE, seed=SEED, drop_last=True)
sampler = GroupedBatchSampler(neighbors_idx, cfg)
batches = sampler.build_batches()

print("Total batches:", len(batches))
print("First batch:", batches[0])
print("Unique samples used:", len(set([i for b in batches for i in b])))

Total batches: 112
First batch: [776, 631, 427, 998, 829, 275, 535, 334]
Unique samples used: 896




## 7) Batch inspection (human sanity check) -- Inspect a few batches

You should see instructions in the same batch that are semantically related.

In [ ]:
def preview_text(i: int, max_len: int = 160) -> str:
    t = split[i][TEXT_FIELD]
    return (t[:max_len] + "…") if len(t) > max_len else t

for bi in range(min(3, len(batches))):
    print("\n" + "="*100)
    print("BATCH", bi, "indices:", batches[bi])
    for idx in batches[bi]:
        print(f"- idx={idx:5d} id={id_map[idx]} :: {preview_text(idx)}")


BATCH 0 indices: [776, 631, 427, 998, 829, 275, 535, 334]
- idx=  776 id=776 :: What are the key elements to a companies income statement?
- idx=  631 id=631 :: What is a compound statement?
- idx=  427 id=427 :: You are a product manager at an enterprise software company Y. Your CEO has asked you to write a report on how well the company supports X capabilities. Provide…
- idx=  998 id=998 :: What are common executive roles at large companies?
- idx=  829 id=829 :: Given this reference text about beneficence, what can I do to ensure compliance to the beneficence concept?
- idx=  275 id=275 :: What are dividends?
- idx=  535 id=535 :: Write a brief paragraph of the benefits of attending Arizona State University
- idx=  334 id=334 :: What are some disadvantages of the way the tax code treats incentive stock options?

BATCH 1 indices: [507, 125, 676, 706, 668, 976, 424, 319]
- idx=  507 id=507 :: Who were the founding members of id Software?
- idx=  125 id=125 :: When and where was the 

## 8) Export Debug Batches (optional) --export some grouped batches for debugging

Writes a small `.jsonl` file to Drive so you can review batch composition later
or attach it as an appendix for your thesis methodology.

In [ ]:
if EXPORT_DEBUG_BATCHES:
    out_path = f"{DIRS['runs']}/experiment_logs/grouped_batches_preview__{BUNDLE_NAME}__bs{BATCH_SIZE}.jsonl"
    _safe_makedirs(os.path.dirname(out_path))

    with open(out_path, "w", encoding="utf-8") as f:
        for bi in range(min(50, len(batches))):
            batch = batches[bi]
            row = {
                "batch_id": bi,
                "indices": batch,
                "ids": [id_map[i] for i in batch],
                "texts": [split[i][TEXT_FIELD] for i in batch],
            }
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("Wrote:", out_path)

Wrote: /content/drive/MyDrive/Compass_AI_ML_Research/llm-batching-research/runs/experiment_logs/grouped_batches_preview__dolly_small_1k__all-MiniLM-L6-v2__bs8.jsonl
